# Reconocimiento de dígitos escritos a mano (MNIST)

Partimos del modelo visto en clase (MLP: 784 → 64 → 64 → 10) y lo mejoramos para que **generalice** en lugar de **memorizar**. Al final se puede subir una foto de un número y el modelo dice cuál es.

**Plan**
1. Preparar los datos: normalizar y separar en entrenamiento / validación / prueba.
2. Entrenar el modelo original y comprobar si memoriza (curvas de aprendizaje).
3. Entrenar el modelo mejorado (Dropout, L2, aumento de datos, parada temprana).
4. Evaluar una sola vez con el conjunto de prueba.
5. Predecir a partir de una foto.

In [7]:
import numpy as np
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

tf.keras.utils.set_random_seed(42)   # resultados reproducibles
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

tf.keras.utils.set_random_seed(42)   # resultados reproducibles

ModuleNotFoundError: No module named 'tensorflow'

## 1. Datos: tres conjuntos distintos

| Conjunto | Tamaño | Para qué sirve |
|---|---|---|
| Entrenamiento | 54 000 | El modelo aprende de estas imágenes |
| Validación | 6 000 | Se usa **durante** el entrenamiento para vigilar si el modelo memoriza y decidir cuándo parar |
| Prueba | 10 000 | Se usa **una sola vez, al final**, para dar la nota real con datos nunca vistos |

El código original solo mostraba la exactitud sobre los datos de entrenamiento, y eso no permite saber si el modelo está memorizando.

In [ ]:
(x_train_full, y_train_full), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Normalizar: pasar de 0-255 a 0-1 (el modelo original no lo hacía; así entrena mejor y más estable)
x_train_full = x_train_full.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Añadir el canal: (28, 28) -> (28, 28, 1). Lo necesitan las capas de aumento de datos.
x_train_full = x_train_full[..., np.newaxis]
x_test = x_test[..., np.newaxis]

# Separar validación (estratificada: misma proporción de cada dígito en ambos grupos)
x_train, x_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full, test_size=0.1, stratify=y_train_full, random_state=42)

print("Entrenamiento:", x_train.shape, "| Validación:", x_val.shape, "| Prueba:", x_test.shape)

plt.imshow(x_train[0].reshape(28, 28), cmap='gray')
plt.title(f"Etiqueta: {y_train[0]}")
plt.show()

## 2. El modelo original: ¿memoriza?

Es exactamente el modelo de clase (64-64, 30 épocas, batch de 32), pero ahora vigilando la validación.

**Cómo leer las gráficas:** si la curva de *entrenamiento* sigue mejorando mientras la de *validación* se estanca o empeora, el modelo está memorizando los ejemplos de entrenamiento (**sobreajuste**). Cuanto mayor sea la separación entre las dos curvas, peor generaliza.

In [ ]:
def graficar_historial(historia, titulo):
    h = historia.history
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(h['loss'], label='entrenamiento')
    ax[0].plot(h['val_loss'], label='validación')
    ax[0].set_title(f'{titulo} - pérdida'); ax[0].set_xlabel('época'); ax[0].legend()
    ax[1].plot(h['accuracy'], label='entrenamiento')
    ax[1].plot(h['val_accuracy'], label='validación')
    ax[1].set_title(f'{titulo} - exactitud'); ax[1].set_xlabel('época'); ax[1].legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Modelo original del profesor (sin cambios), con los datos ya normalizados y aplanados a 784
xb_train = x_train.reshape(len(x_train), 784)
xb_val = x_val.reshape(len(x_val), 784)

In = tf.keras.layers.Input(shape=(784,))
Cp1 = tf.keras.layers.Dense(64, activation='relu')(In)
Cp2 = tf.keras.layers.Dense(64, activation='relu')(Cp1)
Out = tf.keras.layers.Dense(10, activation='softmax')(Cp2)
ModelBase = tf.keras.Model(inputs=In, outputs=Out)

ModelBase.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
hist_base = ModelBase.fit(xb_train, y_train, validation_data=(xb_val, y_val),
                          epochs=30, batch_size=32, verbose=0)

graficar_historial(hist_base, 'Modelo base')
h = hist_base.history
print(f"Última época  -> entrenamiento: {h['accuracy'][-1]*100:.2f}% | validación: {h['val_accuracy'][-1]*100:.2f}%")
print(f"Mejor pérdida de validación en la época {np.argmin(h['val_loss']) + 1} de 30")

## 3. Modelo mejorado

| Técnica | Qué problema evita |
|---|---|
| Normalizar (÷255) | Entrenamiento lento o inestable por valores grandes |
| **Dropout (0.3)** | Apaga neuronas al azar en cada paso: ninguna puede "memorizar" por su cuenta |
| **Regularización L2** | Penaliza pesos muy grandes, que son los que se ajustan al ruido |
| **Aumento de datos** (giro ±18°, desplazamiento y zoom del 10 %) | El modelo no puede memorizar posiciones exactas; ve cada dígito ligeramente distinto en cada época. Además lo prepara para fotos reales |
| **EarlyStopping** | Detiene el entrenamiento cuando la validación deja de mejorar y se queda con los mejores pesos |
| ReduceLROnPlateau | Baja la tasa de aprendizaje cuando la validación se estanca |

Las capas de aumento **solo actúan al entrenar**; al validar o predecir no modifican la imagen.

In [ ]:
# Aumento de datos: el fondo es negro, así que lo que "entra" al mover o girar se rellena con 0
aumento = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.05, fill_mode='constant', fill_value=0.0),
    tf.keras.layers.RandomTranslation(0.1, 0.1, fill_mode='constant', fill_value=0.0),
    tf.keras.layers.RandomZoom(0.1, fill_mode='constant', fill_value=0.0),
], name='aumento_de_datos')

reg = tf.keras.regularizers.l2(1e-4)

In = tf.keras.layers.Input(shape=(28, 28, 1))
Aum = aumento(In)
Pl = tf.keras.layers.Flatten()(Aum)
Cp1 = tf.keras.layers.Dense(128, activation='relu', kernel_regularizer=reg)(Pl)
Cp1 = tf.keras.layers.Dropout(0.3)(Cp1)
Cp2 = tf.keras.layers.Dense(64, activation='relu', kernel_regularizer=reg)(Cp1)
Cp2 = tf.keras.layers.Dropout(0.3)(Cp2)
Out = tf.keras.layers.Dense(10, activation='softmax')(Cp2)
Model = tf.keras.Model(inputs=In, outputs=Out)

Model.summary()

In [ ]:
Model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

callbacks = [
    # Parar cuando la pérdida de validación no mejore en 6 épocas seguidas y volver a los mejores pesos
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
    # Si se estanca durante 2 épocas, reducir a la mitad la tasa de aprendizaje
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5),
    # Guardar en disco el mejor modelo visto
    tf.keras.callbacks.ModelCheckpoint('mejor_modelo.keras', monitor='val_loss', save_best_only=True),
]

historia = Model.fit(x_train, y_train,
                     validation_data=(x_val, y_val),
                     epochs=40, batch_size=128,
                     callbacks=callbacks, verbose=2)

In [ ]:
graficar_historial(historia, 'Modelo mejorado')
h = historia.history
print(f"Épocas entrenadas: {len(h['loss'])}")
print(f"Mejor época (menor pérdida de validación): {np.argmin(h['val_loss']) + 1}")
print(f"Entrenamiento: {h['accuracy'][-1]*100:.2f}% | validación: {h['val_accuracy'][-1]*100:.2f}%")

**Qué comparar con la sección 2:** en el modelo mejorado las dos curvas deben ir mucho más juntas. Una exactitud de entrenamiento algo menor que la del modelo base **no es un problema**: es la señal de que ya no memoriza.

## 4. Evaluación final (datos nunca vistos)

El conjunto de prueba no se ha usado para entrenar, ni para validar, ni para elegir cuándo parar. Esta es la nota real del modelo.

In [ ]:
perdida, exactitud = Model.evaluate(x_test, y_test, verbose=0)
print(f"Exactitud en PRUEBA: {exactitud*100:.2f}%")

y_pred = np.argmax(Model.predict(x_test, verbose=0), axis=1)
print()
print(classification_report(y_test, y_pred, digits=4))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6.5, 5.5))
plt.imshow(cm, cmap='Blues')
plt.colorbar()
plt.xticks(range(10)); plt.yticks(range(10))
plt.xlabel('Predicho'); plt.ylabel('Real'); plt.title('Matriz de confusión')
for i in range(10):
    for j in range(10):
        plt.text(j, i, cm[i, j], ha='center', va='center', fontsize=7,
                 color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.show()

In [ ]:
# ¿En qué se equivoca? (los primeros 10 errores)
errores = np.where(y_pred != y_test)[0]
print(f"{len(errores)} errores de {len(y_test)} imágenes de prueba")

fig, ax = plt.subplots(2, 5, figsize=(10, 4.4))
for a, i in zip(ax.ravel(), errores[:10]):
    a.imshow(x_test[i].reshape(28, 28), cmap='gray')
    a.set_title(f"real {y_test[i]} / pred {y_pred[i]}", fontsize=9)
    a.axis('off')
plt.tight_layout()
plt.show()

## 5. Reconocer un número a partir de una foto

Una foto real **no se parece** a una imagen de MNIST: tiene color, fondo claro, sombras, ruido y el número ocupa un tamaño cualquiera. MNIST son dígitos **claros sobre fondo negro**, de 28×28, dentro de una caja de 20×20 y centrados por su centro de masa. Si le damos la foto tal cual, el modelo falla aunque esté bien entrenado.

`preparar_imagen` convierte la foto al formato MNIST:
1. Pasa a gris y decide si hay que **invertir** (tinta oscura sobre papel claro).
2. **Quita sombras y degradados de luz** (transformada top-hat).
3. Separa el trazo del fondo con el umbral de **Otsu** y descarta motas sueltas.
4. **Recorta** el dígito, lo escala a 20×20 conservando la proporción y lo **centra** en 28×28.

In [ ]:
import numpy as np
from PIL import Image, ImageOps
from scipy import ndimage


def _umbral_otsu(a):
    """Umbral de Otsu para una imagen en [0,1]: separa 'trazo' de 'fondo'."""
    hist, bordes = np.histogram(a.ravel(), bins=256, range=(0.0, 1.0))
    p = hist / max(hist.sum(), 1)
    centros = (bordes[:-1] + bordes[1:]) / 2
    w0 = np.cumsum(p)
    w1 = 1.0 - w0
    m0 = np.cumsum(p * centros)
    mt = m0[-1]
    with np.errstate(divide="ignore", invalid="ignore"):
        varianza = (mt * w0 - m0) ** 2 / (w0 * w1)
    varianza = np.nan_to_num(varianza)
    return float(centros[np.argmax(varianza)])


def preparar_imagen(ruta):
    """Convierte una foto de un número al formato MNIST: 28x28, float32 en [0,1],
    dígito claro sobre fondo negro, centrado por centro de masa."""
    img = ImageOps.exif_transpose(Image.open(ruta)).convert("L")   # gris (y corrige giro del celular)
    img.thumbnail((800, 800), Image.LANCZOS)                       # fotos enormes -> tamaño manejable
    a = np.asarray(img, dtype=np.float32) / 255.0

    # MNIST es dígito CLARO sobre fondo OSCURO. En una foto suele ser al revés
    # (tinta oscura sobre papel blanco): miramos el borde de la imagen para decidir.
    borde = np.concatenate([a[0, :], a[-1, :], a[:, 0], a[:, -1]])
    if np.median(borde) > 0.5:
        a = 1.0 - a

    # Quitar iluminación desigual (sombras, degradados): top-hat = imagen - su "fondo".
    # El fondo se estima con una apertura morfológica más ancha que el grosor del trazo.
    a = ndimage.gaussian_filter(a, 1.0)                            # suaviza el ruido de la cámara
    a = ndimage.white_tophat(a, size=max(15, max(a.shape) // 8))
    if a.max() < 0.12:
        raise ValueError("Casi no hay contraste: no se ve ningún dígito. "
                         "Usa un trazo grueso, fondo liso y buena luz.")
    a = a / a.max()

    # Separar trazo y fondo con el umbral de Otsu y descartar manchas pequeñas sueltas
    mascara = a > _umbral_otsu(a)
    etiquetas, n = ndimage.label(mascara)
    if n == 0:
        raise ValueError("No se detectó ningún dígito en la imagen.")
    areas = np.bincount(etiquetas.ravel())[1:]
    validas = np.where(areas >= max(10, 0.05 * areas.max()))[0] + 1
    mascara = np.isin(etiquetas, validas)
    a = np.where(mascara, a, 0.0)

    # Recortar al rectángulo que contiene el dígito
    filas = np.where(mascara.any(axis=1))[0]
    cols = np.where(mascara.any(axis=0))[0]
    recorte = a[filas[0]:filas[-1] + 1, cols[0]:cols[-1] + 1]

    # Igual que en MNIST: el dígito cabe en una caja de 20x20 conservando la proporción
    h, w = recorte.shape
    escala = 20.0 / max(h, w)
    nw, nh = max(1, round(w * escala)), max(1, round(h * escala))
    pequeno = Image.fromarray((recorte * 255).astype(np.uint8)).resize((nw, nh), Image.LANCZOS)

    lienzo = np.zeros((28, 28), dtype=np.float32)
    y0, x0 = (28 - nh) // 2, (28 - nw) // 2
    lienzo[y0:y0 + nh, x0:x0 + nw] = np.asarray(pequeno, dtype=np.float32) / 255.0

    # ...y centrado por centro de masa (así se centró MNIST)
    cy, cx = ndimage.center_of_mass(lienzo)
    lienzo = ndimage.shift(lienzo, (13.5 - cy, 13.5 - cx), order=1, mode="constant", cval=0.0)
    lienzo = np.clip(lienzo, 0.0, 1.0)
    if lienzo.max() > 0:
        lienzo = lienzo / lienzo.max()
    return lienzo.astype(np.float32)

In [ ]:
from PIL import Image, ImageOps


def predecir_foto(ruta, modelo=None):
    if modelo is None:
        modelo = Model
    img28 = preparar_imagen(ruta)
    prob = modelo.predict(img28[np.newaxis, ..., np.newaxis], verbose=0)[0]
    pred = int(np.argmax(prob))

    fig, ax = plt.subplots(1, 3, figsize=(12, 3.4))
    ax[0].imshow(ImageOps.exif_transpose(Image.open(ruta)), cmap='gray')
    ax[0].set_title('Foto original'); ax[0].axis('off')
    ax[1].imshow(img28, cmap='gray')
    ax[1].set_title('Lo que ve el modelo (28x28)'); ax[1].axis('off')
    ax[2].bar(range(10), prob)
    ax[2].set_xticks(range(10)); ax[2].set_ylim(0, 1)
    ax[2].set_title('Probabilidad por dígito')
    plt.tight_layout()
    plt.show()

    print(f"Número predicho: {pred}  (confianza {prob[pred]*100:.1f}%)")
    otros = [f"{d}: {prob[d]*100:.1f}%" for d in np.argsort(prob)[::-1][1:3]]
    print("Siguientes candidatos ->", " | ".join(otros))
    if prob[pred] < 0.6:
        print("Aviso: el modelo no está seguro. Prueba con un trazo más grueso, fondo liso y mejor luz.")
    return pred

In [ ]:
# Prueba del procesamiento SIN foto: simulamos una foto (dígito oscuro sobre papel blanco, ampliado a 280x280)
Image.fromarray((255 - x_test[0, :, :, 0] * 255).astype(np.uint8)).resize((280, 280), Image.BICUBIC).save('prueba.png')
predecir_foto('prueba.png')
print("Etiqueta real:", y_test[0])

In [ ]:
# Sube tu propia foto
try:
    from google.colab import files
    subido = files.upload()              # elige la foto de tu número
    ruta_foto = next(iter(subido))
except ImportError:
    ruta_foto = 'mi_numero.png'          # fuera de Colab: escribe aquí la ruta de tu foto

predecir_foto(ruta_foto)

In [ ]:
# Guardar el modelo entrenado (con los mejores pesos) para usarlo después sin reentrenar
Model.save('modelo_mnist.keras')

# Para cargarlo más tarde:
# Model = tf.keras.models.load_model('modelo_mnist.keras')

## 6. Conclusiones

**Cómo se evitó la memorización**
- Se separó un conjunto de **validación** para detectar el sobreajuste mientras se entrena.
- **Dropout** y **regularización L2** limitan la capacidad del modelo de ajustarse al ruido.
- El **aumento de datos** hace que cada época vea imágenes ligeramente distintas.
- **EarlyStopping** detiene el entrenamiento en el mejor punto y recupera esos pesos.
- El conjunto de **prueba** se usó una sola vez, al final.

**Resultados** (completar con los valores obtenidos)
- Modelo base: entrenamiento ___ % / validación ___ %
- Modelo mejorado: entrenamiento ___ % / validación ___ % / prueba ___ %

**Consejos para que la foto funcione bien**
- Número escrito con rotulador o marcador **grueso**, sobre papel liso.
- **Un solo dígito**, ocupando buena parte de la foto y sin otras cosas alrededor.
- Buena luz, sin sombras fuertes encima del número.